# Domain check — wedge vs full disk

The same integrator on two domains, plus three quantitative checks with definite right answers: φ-decomposition (sub-wedges sum to the whole), photospheric clip (EM jumps up crossing the limb), and azimuthal symmetry (pB constant on a circle of fixed impact parameter).

In [ ]:
import numpy as np
from rrt import synthdata
from rrt.geometry import prepare_domain
from rrt.observer import make_observer, image_grid
from rrt.euv import run_siddon
from rrt.wl import run_siddon_pB
from rrt.response import FLAT_T, FLAT_R
_, e_los, x_img, y_img = make_observer(-70, 0)
def em(Xg, Yg, r, th, ph, ne, T, npix):
    return run_siddon(Xg, Yg, x_img, y_img, e_los, r, th, ph, ne, T,
                      FLAT_T, FLAT_R, Npix=npix, Rmax=3.0, plot=False, save=False)

In [ ]:
for dom in ('wedge', 'full'):
    r, th, ph = synthdata.spherical_grid(dom, nr=10, nth=12, nph=16, r_range=(1.0, 3.0))
    d = prepare_domain(r, th, ph)
    print(f'{dom:6s} full_theta={d["full_theta"]} full_phi={d["full_phi"]} phi_span={np.degrees(d["phi_span"]):.0f}deg')

In [ ]:
# Check 1 — phi decomposition on the full domain
r, th, ph = synthdata.spherical_grid('full', nr=10, nth=12, nph=16, r_range=(1.0,3.0))
ne = synthdata.analytic_ne(r, th, ph, aniso=True); T = synthdata.analytic_T(r, th, ph)
Xg, Yg, _ = image_grid(3.0, 48)
whole = em(Xg, Yg, r, th, ph, ne, T, 48)
parts = np.zeros_like(whole)
for k in range(4):
    s = slice(k*ph.size//4, (k+1)*ph.size//4)
    parts += em(Xg, Yg, r, th, ph[s], np.ascontiguousarray(ne[:,:,s]),
                np.ascontiguousarray(T[:,:,s]), 48)
rel = np.abs(parts-whole).sum()/np.abs(whole).sum()
print(f'phi-decomposition rel.diff = {rel:.2e}  ->  {"PASS" if rel<2e-2 else "FAIL"}')